### Step 1: Project Setup

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY    = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY      = os.getenv("OPENROUTER_API_KEY")

print("Gemini key loaded:       ", "✅" if GEMINI_API_KEY    else "❌ Missing!")
print("Groq key loaded:         ", "✅" if GROQ_API_KEY      else "❌ Missing!")
print("OpenRouter key loaded:   ", "✅" if OPENROUTER_API_KEY      else "❌ Missing!")

Gemini key loaded:        ✅
Groq key loaded:          ✅
OpenRouter key loaded:    ✅


In [2]:
from google import genai
from groq import Groq

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ GEMINI client ready")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ GROQ client ready")

✅ GEMINI client ready
✅ GROQ client ready


### Step 2: Index creation without any LLM

#### PageIndex's underlying Flash function without LLM summary

Instead of running PageIndexClient.submit_document() (which forces LLM summarization on every node), call PageIndex's underlying Flash function directly with summary=False. Using optimize="merge" (deterministic, LLM-free pass) or optimize=False builds the entire hierarchical tree purely from the PDF layout and bookmarks in seconds with zero Ollama calls and zero API rate limits.

In [ ]:
import json
import time
from pathlib import Path
import fitz
import pymupdf
from pageindex.flash import page_index_flash


def build_fast_legal_index(corpus_dir: str = "../corpus", output_dir: str = "../legal_index",):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    manifest = {}

    for pdf_path in sorted(Path(corpus_dir).glob("*.pdf")):
        t0 = time.perf_counter()
        print(f"[INDEXING] {pdf_path.name} (zero-LLM Flash mode) ...")

        # Pure deterministic layout/bookmark tree - zero LLM calls
        tree = page_index_flash(
            str(pdf_path),
            summary=False,
            optimize="merge",  # use optimize=False if raw unmerged nodes is needed
        )

        # Save hierarchical tree JSON
        tree_file = out / f"{pdf_path.stem}_tree.json"
        tree_file.write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")

        # Extract & cache 1-indexed verbatim pages once for <5ms retrieval in LangGraph
        doc = fitz.open(pdf_path)
        pages = {str(i + 1): page.get_text("text") for i, page in enumerate(doc)}
        doc.close()

        pages_file = out / f"{pdf_path.stem}_pages.json"
        pages_file.write_text(json.dumps(pages, indent=2, ensure_ascii=False), encoding="utf-8")

        manifest[pdf_path.stem] = {
            "source_pdf": pdf_path.name,
            "page_count": len(pages),
            "tree_file": str(tree_file),
            "pages_file": str(pages_file),
        }
        print(f"[DONE] {pdf_path.name} ({len(pages)} pages) in {time.perf_counter() - t0:.2f}s")

    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nAll statutes indexed. Manifest saved to {out / 'manifest.json'}")


if __name__ == "__main__":
    build_fast_legal_index()

### Step 3: Tree Audit (top level view) for all the documents tree.json files

- bnsAct2023_tree

In [16]:
import json
from pathlib import Path

tree_path = Path("../legal_index/bnsAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )

Doc: bnsAct2023.pdf | Title:  Corresponding Section Table of BNS with Repealed Act

=== TOP-LEVEL NODES ===
[0000] pp. 1-2 | children=0 | homepage
[0001] pp. 2-20 | children=18 | index
    └─ [0002] pp. 2-2 | CHAPTER I PRELIMINARY; CHAPTER II OF PUNISHMENTS
    └─ [0003] pp. 2-4 | CHAPTER III GENERAL EXCEPTIONS
    └─ [0004] pp. 4-5 | CHAPTER IV OF ABETMENT, CRIMINAL CONSPIRACY AND ATTEMPT
    └─ [0005] pp. 5-7 | CHAPTER V OF OFFENCES AGAINST WOMAN AND CHILD
    └─ [0006] pp. 7-9 | CHAPTER VI OF OFFENCES AFFECTING THE HUMAN BODY
    └─ [0007] pp. 9-9 | CHAPTER VII OF OFFENCES AGAINST THE STATE
    └─ [0008] pp. 9-10 | CHAPTER VIII OF OFFENCES RELATING TO THE ARMY, NAVY AND AIR FORCE
    └─ [0009] pp. 10-10 | CHAPTER IX OF OFFENCES RELATING TO ELECTIONS
    └─ [0010] pp. 10-11 | CHAPTER X OF OFFENCES RELATING TO COIN, CURRENCY-NOTES, BANK-NOTES, AND GOVERNMENT STAMPS
    └─ [0011] pp. 11-11 | CHAPTER XI OF OFFENCES AGAINST THE PUBLIC TRANQUILLITY
    └─ [0012] pp. 11-12 | CHAPTER XII O

- bnssAct2023_tree

In [17]:
import json
from pathlib import Path

tree_path = Path("../legal_index/bnssAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: bnssAct2023.pdf | Title: bl Hkkx esa fHkUu i`"B la[;k nh tkrh gS ftlls fd ;g vyx ladyu ds :i esa j[kk tk ldsA

=== TOP-LEVEL NODES ===
[0000] pp. 1-4 | children=0 | Preface
[0001] pp. 4-9 | children=0 | CHAPTER II 
[0002] pp. 9-11 | children=0 | CHAPTER III 
[0003] pp. 11-13 | children=0 | CHAPTER IV 
[0004] pp. 13-20 | children=0 | CHAPTER V 
[0005] pp. 20-26 | children=4 | CHAPTER VI 
    └─ [0006] pp. 20-21 | A. —Summons
    └─ [0007] pp. 21-24 | B. —Warrant of arrest
    └─ [0008] pp. 24-26 | C. —Proclamation and attachment
    └─ [0009] pp. 26-26 | D. —Other rules regarding processes
[0010] pp. 26-32 | children=4 | CHAPTER VII 
    └─ [0011] pp. 26-27 | A. —Summons to produce
    └─ [0012] pp. 27-29 | B. —Search-warrants
    └─ [0013] pp. 29-30 | C. —General provisions relating to searches
    └─ [0014] pp. 30-32 | D. —Miscellaneous
[0015] pp. 32-36 | children=0 | CHAPTER VIII 
[0016] pp. 36-42 | children=0 | CHAPTER IX 
[0017] pp. 42-45 | children=0 | CHAPTER X 
[0018] pp. 4

- constitutionOfIndia1950

In [18]:
import json
from pathlib import Path

tree_path = Path("../legal_index/constitutionOfIndia1950_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: constitutionOfIndia1950.pdf | Title: भारत का संविधान

=== TOP-LEVEL NODES ===
[0000] pp. 1-1 | children=0 | Preface
[0001] pp. 2-2 | children=0 | PREFACE
[0002] pp. 3-4 | children=0 | LIST OF ABBREVIATIONS USED
[0003] pp. 5-6 | children=0 | PART III
[0004] pp. 6-7 | children=0 | PART IV
[0005] pp. 7-7 | children=0 | PART IVA
[0006] pp. 7-12 | children=6 | PART V
    └─ [0007] pp. 7-8 | The President and Vice-President
    └─ [0008] pp. 8-8 | Council of Ministers; The Attorney-General for India
    └─ [0009] pp. 9-9 | Conduct of Government Business
    └─ [0010] pp. 9-11 | CHAPTER II .⎯PARLIAMENT
    └─ [0011] pp. 11-11 | CHAPTER III .⎯LEGISLATIVE POWERS OF THE PRESIDENT
    └─ [0012] pp. 11-12 | CHAPTER IV .⎯ THE UNION JUDICIARY
[0013] pp. 12-13 | children=0 | PART VI
[0014] pp. 13-13 | children=0 | CHAPTER II .⎯ THE EXECUTIVE
[0015] pp. 13-17 | children=5 | CHAPTER III .⎯THE STATE LEGISLATURE
    └─ [0016] pp. 13-14 | General
    └─ [0017] pp. 14-14 | Officers of the State Legisl

- dpdpAct2023

In [19]:
import json
from pathlib import Path

tree_path = Path("../legal_index/dpdpAct2023_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: dpdpAct2023.pdf | Title: bl Hkkx esa fHkUu i`"B la[;k nh tkrh gS ftlls fd ;g vyx ladyu ds :i esa j[kk tk ldsA

=== TOP-LEVEL NODES ===
[0000] pp. 1-4 | children=0 | Preface
[0001] pp. 4-9 | children=3 | CHAPTER II 
    └─ [0002] pp. 4-6 | OBLIGATIONS OF DATA FIDUCIARY
    └─ [0003] pp. 6-8 | Illustrations.
    └─ [0004] pp. 8-9 | Illustrations.
[0005] pp. 9-10 | children=0 | CHAPTER III 
[0006] pp. 11-12 | children=0 | CHAPTER IV 
[0007] pp. 12-14 | children=0 | CHAPTER V 
[0008] pp. 14-15 | children=0 | CHAPTER VI 
[0009] pp. 15-16 | children=0 | CHAPTER VII 
[0010] pp. 16-17 | children=0 | CHAPTER VIII 
[0011] pp. 17-20 | children=0 | CHAPTER IX 
[0012] pp. 21-21 | children=0 | THE SCHEDULE


- itActAct2020

In [40]:
import json
from pathlib import Path

tree_path = Path("../legal_index/itAct2020_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))

print(f"Doc: {data.get('doc_name')} | Title: {data.get('doc_title')}\n")
print("=== TOP-LEVEL NODES ===")
for node in data.get("structure", []):
    children = node.get("nodes", [])
    print(
        f"[{node['node_id']}] pp. {node['start_index']}-{node['end_index']} "
        f"| children={len(children)} | {node['title']}"
    )
    # Preview all children of each non-empty top-level section
    for child in children[:]:
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| {child['title']}"
        )
        

Doc: itAct2020.pdf | Title: Information Technology Act, 2000 (IT Act)

=== TOP-LEVEL NODES ===
[0000] pp. 1-2 | children=0 | Preface
[0001] pp. 2-2 | children=0 | CHAPTER VII ; ELECTRONIC SIGNATURE CERTIFICATES; CHAPTER VIII ; DUTIES OF SUBSCRIBERS; CHAPTER IX ; PENALTIES, COMPENSATION AND ADJUDICATION
[0002] pp. 2-3 | children=0 | CHAPTER X 
[0003] pp. 3-3 | children=0 | p.3
[0004] pp. 3-5 | children=0 | CHAPTER XIII MISCELLANEOUS
[0005] pp. 5-8 | children=0 | CHAPTER 1 PRELIMINARY
[0006] pp. 8-9 | children=0 | CHAPTER II 
[0007] pp. 9-11 | children=0 | CHAPTER III 
[0008] pp. 11-12 | children=0 | CHAPTER IV 
[0009] pp. 12-12 | children=0 | CHAPTER V 
[0010] pp. 12-23 | children=5 | CHAPTER VI 
    └─ [0011] pp. 12-16 | REGULATION OF CERTIFYING AUTHORITIES
    └─ [0012] pp. 16-18 | 1 [ELECTRONIC SIGNATURE] CERTIFICATES
    └─ [0013] pp. 18-18 | CHAPTER VIII 
    └─ [0014] pp. 18-21 | CHAPTER IX 
    └─ [0015] pp. 21-23 | CHAPTER X 
[0016] pp. 23-30 | children=0 | CHAPTER XI 
[0017] pp

### Step 4: Get Summary to the nodes of JSON

Only the root nodes and 1 child node will be generating the summary from LLM based on:
- node title
- node information stores in page.json

In [45]:
# Count the total no. of summaries that will be generated before actually generating it. (to save time)
# To actually generate summaries for nodes
# For long sections, it was using only the few page from start, but then that summary will be incomplete.
# tree nodes already contain all sub-section titles inside node["nodes"] and node["key_items"]. 
# Feeding those sub-headings plus the opening page text gives the LLM visibility across all 25 pages while staying well within Groq's 8,000 TPM limit


import json
import re
import time
from pathlib import Path
import litellm
from dotenv import load_dotenv

load_dotenv()

MODEL = "groq/openai/gpt-oss-20b"   # Free tier: 30 RPM, 8K TPM, 1K RPD
DRY_RUN = True                     # 1) Run True to check counts, 2) Set False to generate
CONSTITUTION_BODY_START = 32        # actual start

SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):  # Skip BNSS Second Schedule forms
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def get_summary_targets(structure: list, doc_key: str) -> list:
    targets = []
    for node in structure:
        if should_skip(node, doc_key):
            continue
        title = node.get("title", "").strip().lower()
        span = int(node.get("end_index") or 1) - int(node.get("start_index") or 1)
        if node.get("nodes") and span > 40 and title != "compare":
            targets.extend(c for c in node["nodes"] if not should_skip(c, doc_key))
        else:
            targets.append(node)
    return targets


litellm.drop_params = True

def summarize(title: str, text: str) -> str:
    resp = litellm.completion(
        model=MODEL,
        temperature=0,
        max_tokens=512,
        reasoning_effort="low",
        num_retries=5,
        messages=[{
            "role": "user",
            "content": (
                f"Write a concise 1-2 sentence plain-text summary of this Indian legal section ('{title}'), "
                f"listing key topics, offences, procedures, or rights:\n\n{text[:2500]}"
            ),
        }],
    )
    out = (resp.choices[0].message.content or "").strip()
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()

def build_summary_context(node: dict, pages: dict, max_chars: int = 2500) -> str:
    # Collect immediate child titles / key_items so long chapters aren't blind to later sections
    subtopics = []
    for c in node.get("nodes", []):
        if c.get("title") and not c["title"].startswith("p."):
            subtopics.append(c["title"])
        subtopics.extend(k for k in c.get("key_items", []) if not k.startswith("p."))
    subtopics.extend(k for k in node.get("key_items", []) if not k.startswith("p."))

    header = f"Sub-sections in this chapter: {'; '.join(dict.fromkeys(subtopics))[:1200]}\n\n" if subtopics else ""

    s = int(node.get("start_index") or 1)
    e = int(node.get("end_index") or s)
    body = "\n".join(pages.get(str(p), "") for p in range(s, e + 1))
    return (header + body)[:max_chars]

def add_summaries(index_dir: str = "../legal_index"):
    base = Path(index_dir) if Path(index_dir).exists() else Path("./legal_index")
    manifest = json.loads((base / "manifest.json").read_text(encoding="utf-8"))

    for doc_key, meta in manifest.items():
        tree_path = base / Path(meta["tree_file"]).name
        tree = json.loads(tree_path.read_text(encoding="utf-8"))
        pages = json.loads((base / Path(meta["pages_file"]).name).read_text(encoding="utf-8"))

        targets = get_summary_targets(tree.get("structure", []), doc_key)
        for n in targets:
            s = n.get("summary", "").strip()
            if not s or not s.endswith("."):
                n.pop("summary", None)

        pending = [n for n in targets if not n.get("summary")]
        print(f"{doc_key}: {len(targets)} targets ({len(pending)} remaining)")

        if DRY_RUN or not pending:
            continue

        for i, node in enumerate(pending, 1):
            s = int(node.get("start_index") or 1)
            e = int(node.get("end_index") or s)
            snippet = build_summary_context(node, pages) # every chapter summary will then cover all topics from its first page to its last page
            if snippet.strip():
                node["summary"] = summarize(node.get("title", ""), snippet)
                print(f"  [{i}/{len(pending)}] {node['node_id']}: {node['summary'][:80]}...")
                time.sleep(6.5)

        tree_path.write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"  -> Saved {tree_path.name}")

if __name__ == "__main__":
    add_summaries()

    
# Output:

# bnsAct2023: 20 summary targets
# bnssAct2023: 84 summary targets
# constitutionOfIndia1950: 68 summary targets
# dpdpAct2023: 10 summary targets
# itAct2020: 13 summary targets

# bnsAct2023: 20 targets (20 remaining)
# bnssAct2023: 26 targets (26 remaining)
# constitutionOfIndia1950: 43 targets (43 remaining)
# dpdpAct2023: 9 targets (9 remaining)
# itAct2020: 12 targets (12 remaining)

bnsAct2023: 20 targets (0 remaining)
bnssAct2023: 26 targets (0 remaining)
constitutionOfIndia1950: 43 targets (0 remaining)
dpdpAct2023: 9 targets (0 remaining)
itAct2020: 12 targets (0 remaining)


### Step 5: Tree Combining

#### Problem with this approach

Combining all five of the indexed Acts (bnsAct2023, bnssAct2023, constitutionOfIndia1950, dpdpAct2023, and itAct2020) requires solving two issues that single-document tutorials skip:   
- node_id collisions: Every *_tree.json starts at "0000". Merging them without namespacing (doc_key::node_id, e.g., bnsAct2023::0148) causes cross-Act lookup collisions.
- Token overflow: Passing all five complete trees into one LLM call blows past free-tier context/TPM limits. 

**Before merging lets view the tree nodes**

#### View the Headings with Page Number from the JSON tree for [Constitution, BNSS]

In [47]:
import json
from pathlib import Path
import re

CONSTITUTION_BODY_START = 32        # actual start
SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def get_summary_targets(structure: list, doc_key: str) -> list:
    targets = []
    for node in structure:
        if should_skip(node, doc_key):
            continue
        title = node.get("title", "").strip().lower()
        span = int(node.get("end_index") or 1) - int(node.get("start_index") or 1)
        if node.get("nodes") and span > 40 and title != "compare":
            targets.extend(c for c in node["nodes"] if not should_skip(c, doc_key))
        else:
            targets.append(node)
    return targets

for doc in ["bnssAct2023", "constitutionOfIndia1950"]:
    tree_path = Path(f"../legal_index/{doc}_tree.json")
    if not tree_path.exists():
        tree_path = Path(f"./legal_index/{doc}_tree.json")

    tree = json.loads(tree_path.read_text(encoding="utf-8"))
    targets = get_summary_targets(tree.get("structure", []), doc)
    print(f"\n=== {doc} ({len(targets)} targets) ===")
    for t in targets[:20]:
        print(f"  [{t['node_id']}] pp. {t.get('start_index')}-{t.get('end_index')} | {t.get('title')}")
    if len(targets) > 20:
        print(f"  ... and {len(targets) - 20} more (last: {targets[-1].get('title')})")


=== bnssAct2023 (26 targets) ===
  [0001] pp. 4-9 | CHAPTER II 
  [0002] pp. 9-11 | CHAPTER III 
  [0003] pp. 11-13 | CHAPTER IV 
  [0004] pp. 13-20 | CHAPTER V 
  [0005] pp. 20-26 | CHAPTER VI 
  [0010] pp. 26-32 | CHAPTER VII 
  [0015] pp. 32-36 | CHAPTER VIII 
  [0016] pp. 36-42 | CHAPTER IX 
  [0017] pp. 42-45 | CHAPTER X 
  [0018] pp. 45-52 | CHAPTER XI 
  [0023] pp. 52-52 | CHAPTER XII 
  [0024] pp. 52-65 | CHAPTER XIII 
  [0025] pp. 65-67 | CHAPTER XIV 
  [0026] pp. 67-73 | CHAPTER XV 
  [0027] pp. 73-74 | CHAPTER XVI 
  [0028] pp. 74-76 | CHAPTER XVII 
  [0029] pp. 76-82 | CHAPTER XVIII 
  [0032] pp. 82-84 | CHAPTER XIX 
  [0034] pp. 84-85 | A. —Cases instituted on a police report
  [0035] pp. 85-86 | B. —Cases instituted otherwise than on police report
  ... and 6 more (last: WHERE A BOND WITH SURETIES IS TO BE EXECUTED, ADD—)

=== constitutionOfIndia1950 (43 targets) ===
  [0036] pp. 32-32 | THE CONSTITUTION OF INDIA
  [0037] pp. 33-34 | PART I
  [0038] pp. 35-36 | PART II
 

#### View the Child Nodes from the JSON file

In [49]:
# View the Child Nodes from the JSON file

import json
from pathlib import Path

tree = json.loads(Path("../legal_index/constitutionOfIndia1950_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("../legal_index/bnssAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("../legal_index/bnsAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("../legal_index/dpdpAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

tree = json.loads(Path("../legal_index/itAct2020_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

[0000] pp. 1-1 | children=0 | Preface
[0001] pp. 2-2 | children=0 | PREFACE
[0002] pp. 3-4 | children=0 | LIST OF ABBREVIATIONS USED
[0003] pp. 5-6 | children=0 | PART III
[0004] pp. 6-7 | children=0 | PART IV
[0005] pp. 7-7 | children=0 | PART IVA
[0006] pp. 7-12 | children=6 | PART V
[0013] pp. 12-13 | children=0 | PART VI
[0014] pp. 13-13 | children=0 | CHAPTER II .⎯ THE EXECUTIVE
[0015] pp. 13-17 | children=5 | CHAPTER III .⎯THE STATE LEGISLATURE
[0021] pp. 17-17 | children=0 | CHAPTER VI .⎯SUBORDINATE COURTS; PART VIII
[0022] pp. 17-18 | children=0 | PART IX
[0023] pp. 18-19 | children=0 | PART IXA
[0024] pp. 19-20 | children=0 | PART X
[0025] pp. 20-21 | children=0 | PART XI
[0026] pp. 21-23 | children=0 | PART XII
[0027] pp. 23-23 | children=0 | PART XIII
[0028] pp. 23-25 | children=0 | PART XIV
[0029] pp. 25-25 | children=0 | PART XV
[0030] pp. 25-26 | children=0 | PART XVI
[0031] pp. 26-27 | children=0 | PART XVII
[0032] pp. 27-28 | children=0 | PART XVIII
[0033] pp. 28-30 | c

In [51]:
import json
from pathlib import Path

tree = json.loads(Path("../legal_index/bnsAct2023_tree.json").read_text(encoding="utf-8"))
for n in tree.get("structure", []):
    print(f"[{n['node_id']}] pp. {n.get('start_index')}-{n.get('end_index')} | children={len(n.get('nodes', []))} | {n.get('title')}")

[0000] pp. 1-2 | children=0 | homepage
[0001] pp. 2-20 | children=18 | index
[0024] pp. 20-74 | children=24 | Compare
[0067] pp. 74-237 | children=19 | chapters


#### Merging the JSON files using Namespace

In [ ]:
# Merge all the index json files to generate a single JSON file

import json
from pathlib import Path
import re

CONSTITUTION_BODY_START = 32        # actual start
SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)

def prefix_ids(nodes: list, doc_key: str) -> list:
    out = []
    for node in nodes:
        if should_skip(node, doc_key):
            continue
        node["node_id"] = f"{doc_key}::{node['node_id']}"
        if node.get("nodes"):
            node["nodes"] = prefix_ids(node["nodes"], doc_key)
        out.append(node)
    return out

CONSTITUTION_BODY_START = 32       # actual start

SKIP_KEYWORDS = ("homepage", "index", "table of contents", "contents", "preface", "abbreviations")
MIN_BODY_PAGE = {"constitutionOfIndia1950": CONSTITUTION_BODY_START}

def should_skip(node: dict, doc_key: str) -> bool:
    title = node.get("title", "").strip().lower()
    start_p = int(node.get("start_index") or 1)
    if any(k in title for k in SKIP_KEYWORDS):
        return True
    if re.match(r"^form\s+no\.?", title):  # Skip BNSS Second Schedule forms
        return True
    return start_p < MIN_BODY_PAGE.get(doc_key, 1)


def merge_trees(index_dir: str = "./legal_index"):
    base = Path(index_dir)
    manifest = json.loads((base / "manifest.json").read_text(encoding="utf-8"))
    merged = []

    for doc_key, meta in manifest.items():
        tree = json.loads(Path(meta["tree_file"]).read_text(encoding="utf-8"))
        merged.append({
            "node_id": f"{doc_key}::ROOT",
            "title": tree.get("doc_title") or doc_key,
            "start_index": 1,
            "end_index": meta["page_count"],
            "nodes": prefix_ids(tree.get("structure", []), doc_key),
        })

    out_path = base / "merged_tree.json"
    out_path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Merged {len(merged)} Acts -> {out_path}")

if __name__ == "__main__":
    merge_trees()

#### Why this Objection Actually Strengthens my Research
Instead of hiding the LLM-router dependency, make it your central ablation study. In your paper or technical report, compare four configurations on the same 50–100 legal query benchmark:

- Baseline A: Naive Vector RAG (fixed chunking + embeddings).

- **Baseline B: Pure LLM Tree Routing (PageIndex-style only - current revisedmain.ipynb)**.

- Variant C: Hybrid Vectorless (LLM Tree Router + SQLite FTS5/BM25 Union + Deterministic Cross-Ref Expansion).

- Full System (Ours): Variant C + Cyclic LangGraph Completeness Loop & Fail-Closed Verifier.

Measuring Statutory Recall, Proviso/Exception Capture Rate, Cross-Act Recall, and Verbatim Citation Faithfulness across B → C → Full System proves empirically both why separating retrieval from generation prevents hallucination and why deterministic recall guards are necessary to prevent LLM routing omissions.


### Step 6: View the Merged JSON Tree

Those summary=✗ nodes are not missing summaries—their summaries were generated one level deeper on their child nodes because of the span > 40 rule in get_summary_targets.py:

- bnsAct2023::0067 (chapters, pp. 74–237): span = 163 > 40, so all 19 individual chapters inside it (0068 through 0304) received summaries instead of the outer wrapper.

- bnssAct2023::0033 (CHAPTER XX, pp. 84–127): span = 43 > 40, so its sub-chapters (Trial Before a Court of Session, Trial of Warrant-Cases, etc.) were summarized individually.

- constitutionOfIndia1950::0048 (PART V, pp. 57–100): span = 43 > 40, so its chapters (The Executive, Parliament, The Union Judiciary, CAG) received separate summaries.

- constitutionOfIndia1950::0122 (SIXTH SCHEDULE, pp. 311–382): span = 71 > 40, so its sub-nodes were summarized individually.

In [52]:
import json
from pathlib import Path

merged_path = Path("../legal_index/merged_tree.json")
data = json.loads(merged_path.read_text(encoding="utf-8"))

def rollup_summaries(nodes: list):
    for node in nodes:
        children = node.get("nodes", [])
        if children:
            rollup_summaries(children)
            if not node.get("summary"):
                child_titles = [c["title"].strip() for c in children if c.get("title")]
                child_sums = [c["summary"].strip() for c in children if c.get("summary")]
                if child_sums:
                    node["summary"] = f"Covers {', '.join(child_titles[:6])}. " + " ".join(child_sums[:2])

rollup_summaries(data)
merged_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
print("Rolled up child summaries into multi-page parent nodes.")

Rolled up child summaries into multi-page parent nodes.


In [53]:
# Top level view with Summary

import json
from pathlib import Path

tree_path = Path("../legal_index/merged_tree.json")
data = json.loads(tree_path.read_text(encoding="utf-8"))  # list of Act root dicts

print(f"Merged Acts: {len(data)}\n")
print("=== TOP-LEVEL NODES ===")
for act in data:
    chapters = act.get("nodes", [])
    print(
        f"[{act['node_id']}] pp. {act['start_index']}-{act['end_index']} "
        f"| children={len(chapters)} | {act['title']}"
    )
    for child in chapters:
        has_summary = "✓" if child.get("summary") else "✗"
        print(
            f"    └─ [{child['node_id']}] pp. {child['start_index']}-{child['end_index']} "
            f"| summary={has_summary} | {child['title']}"
        )

Merged Acts: 5

=== TOP-LEVEL NODES ===
[bnsAct2023::ROOT] pp. 1-237 | children=2 | Bharatiya Nyaya Sanhita, 2023
    └─ [bnsAct2023::0024] pp. 20-74 | summary=✓ | Compare
    └─ [bnsAct2023::0067] pp. 74-237 | summary=✓ | chapters
[bnssAct2023::ROOT] pp. 1-249 | children=24 | Bharatiya Nagarik Suraksha Sanhita, 2023
    └─ [bnssAct2023::0001] pp. 4-9 | summary=✓ | CHAPTER II 
    └─ [bnssAct2023::0002] pp. 9-11 | summary=✓ | CHAPTER III 
    └─ [bnssAct2023::0003] pp. 11-13 | summary=✓ | CHAPTER IV 
    └─ [bnssAct2023::0004] pp. 13-20 | summary=✓ | CHAPTER V 
    └─ [bnssAct2023::0005] pp. 20-26 | summary=✓ | CHAPTER VI 
    └─ [bnssAct2023::0010] pp. 26-32 | summary=✓ | CHAPTER VII 
    └─ [bnssAct2023::0015] pp. 32-36 | summary=✓ | CHAPTER VIII 
    └─ [bnssAct2023::0016] pp. 36-42 | summary=✓ | CHAPTER IX 
    └─ [bnssAct2023::0017] pp. 42-45 | summary=✓ | CHAPTER X 
    └─ [bnssAct2023::0018] pp. 45-52 | summary=✓ | CHAPTER XI 
    └─ [bnssAct2023::0023] pp. 52-52 | summary=✓ | C

#### View complete tree with summary[:90]

In [54]:
# View complete tree with summary[:90]

import json

with open("../legal_index/merged_tree.json", "r", encoding="utf-8") as f:
    trees = json.load(f)  # list of Act root dicts

def print_tree(node, indent=0):
    prefix = "  " * indent
    node_id = node.get("node_id") or node.get("id") or ""
    title = node.get("title") or node.get("name") or node.get("act_name") or "Untitled"
    summary = (node.get("summary") or "").strip().replace("\n", " ")
    summary_preview = f" — {summary[:90]}..." if len(summary) > 90 else (f" — {summary}" if summary else "")
    print(f"{prefix}├── [{node_id}] {title}{summary_preview}")

    children = node.get("children") or node.get("nodes") or node.get("sections") or []
    for child in children:
        print_tree(child, indent + 1)

print(f"Total Act trees in merged_tree.json: {len(trees)}\n")
for act_tree in trees:
    print_tree(act_tree)

Total Act trees in merged_tree.json: 5

├── [bnsAct2023::ROOT] Bharatiya Nyaya Sanhita, 2023 — Covers Compare, chapters. The section outlines the structure and scope of the Indian Penal...
  ├── [bnsAct2023::0024] Compare — The section outlines the structure and scope of the Indian Penal Code, detailing prelimina...
    ├── [bnsAct2023::0025] Compare (intro)
    ├── [bnsAct2023::0026] CHAPTER II – OF PUNISHMENTS
    ├── [bnsAct2023::0027] CHAPTER III - OF PUNISHMENTS
    ├── [bnsAct2023::0028] CHAPTER III 
    ├── [bnsAct2023::0029] CHAPTER IV 
    ├── [bnsAct2023::0030] CHAPTER IV 
    ├── [bnsAct2023::0031] CHAPTER V 
    ├── [bnsAct2023::0032] CHAPTER V 
    ├── [bnsAct2023::0033] CHAPTER XX 
    ├── [bnsAct2023::0034] CHAPTER VI 
      ├── [bnsAct2023::0035] New Section
      ├── [bnsAct2023::0036] CHAPTER XVI 
        ├── [bnsAct2023::0037] CHAPTER XVI (intro)
        ├── [bnsAct2023::0038] Of wrongful restraint and wrongful confinement
        ├── [bnsAct2023::0039] Of kidnapping

### Step 7: Vectorless RAG Pipeline

Load the merged tree

In [55]:
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

tree_path = Path("./legal_index/merged_tree.json")
if not tree_path.exists():
    tree_path = Path("../legal_index/merged_tree.json")

merged_tree = json.loads(tree_path.read_text(encoding="utf-8"))
print(f"Loaded {len(merged_tree)} statutes from {tree_path}")

Loaded 5 statutes from ..\legal_index\merged_tree.json


Count total no. of nodes

In [62]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(merged_tree)
print(f" ⚽ Total nodes in tree: {total}")
print("     Each node = one retrievable section of the document")

 ⚽ Total nodes in tree: 489
     Each node = one retrievable section of the document


---
#### 🧠 Section A: LLM Tree Search - The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

**Vector RAG retrieval**:
`
    query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
`

*Problem: finds what's similar, not what's relevant*

**PageIndex retrieval**:
`
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
`

*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


#### Finds the relavant nodes

In [79]:
import json
import re
from google import genai
from google.genai import types

EXCLUDED_CONTAINERS = {"homepage", "index", "compare"}


def _normalize_tree(tree) -> list[dict]:
    """Accept merged_tree (list), single *_tree.json (dict with 'structure'), or JSON string."""
    if isinstance(tree, str):
        tree = json.loads(tree)
    if isinstance(tree, dict):
        return tree.get("structure") or tree.get("nodes") or ([tree] if "node_id" in tree else [])
    return tree if isinstance(tree, list) else []


def llm_tree_search(query: str, tree, model: str = "gemini-3.5-flash-lite") -> dict:
    """
    Core PageIndex retrieval compatible with both merged_tree.json (list)
    and individual *_tree.json files (dict with 'structure').
    """
    root_nodes = _normalize_tree(tree)
    valid_ids = set()

    def compress(nodes, depth=0):
        out = []
        for n in nodes:
            if not isinstance(n, dict):
                continue
            title = n.get("title", "").strip()
            title_low = title.lower()
            if title_low in EXCLUDED_CONTAINERS:
                continue
            if title_low == "chapters" and n.get("nodes"):
                out.extend(compress(n["nodes"], depth))
                continue

            nid = n["node_id"]
            valid_ids.add(nid)
            entry = {
                "node_id": nid,
                "title": title,
                "pages": f"{n.get('start_index', '?')}-{n.get('end_index', '?')}",
                "summary": n.get("summary") or ", ".join(n.get("key_items", [])[:6]),
            }
            if depth < 2 and n.get("nodes"):
                entry["children"] = compress(n["nodes"], depth + 1)
            out.append(entry)
        return out

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compress(root_nodes), separators=(",", ":"), ensure_ascii=False)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
        ),
    )

    raw = (response.text or "{}").strip()
    match = re.search(r"[\{\[].*[\}\]]", raw, flags=re.S)
    result = json.loads(match.group(0) if match else raw)

    if isinstance(result, str):
        result = json.loads(result)
    if isinstance(result, list) and result and isinstance(result[0], dict):
        result = result[0]
    if not isinstance(result, dict):
        result = {"thinking": str(result), "node_list": []}

    result["node_list"] = [nid for nid in result.get("node_list", []) if nid in valid_ids]
    return result

In [80]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

🧠 LLM Reasoning:
The user's query is about a lost phone and wallet on a train, and a local police station refusing to register an FIR based on jurisdiction issues. This relates to criminal procedure, specifically the registration of FIRs (First Information Reports) and zero FIRs where jurisdiction cannot be a ground for refusal. However, looking at the provided document tree, the document is the Bharatiya Nyaya Sanhita, 2023 (BNS), which primarily covers substantive criminal law (offences and penalties), whereas procedural matters like FIR registration are handled under the Bharatiya Nagarik Suraksha Sanhita (BNSS). Let's review the chapters: Chapter XII ('Offences by or in relation to public servants') and Chapter XIII ('Offences that obstruct or und

---
#### ⚙️ Section B: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [81]:
import json
import re
from pathlib import Path
from google import genai
from google.genai import types

INDEX_DIR = Path("./legal_index") if Path("./legal_index").exists() else Path("../legal_index")
gemini_client = globals().get("gemini_client") or genai.Client()

_manifest = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
PAGES_STORE = {
    k: json.loads((INDEX_DIR / Path(v["pages_file"]).name).read_text(encoding="utf-8"))
    for k, v in _manifest.items()
}


# ── Helper: Find nodes by ID & attach verbatim page text ─────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk merged_tree.json, collect target_ids, and hydrate text from *_pages.json."""
    found = []

    def walk(nodes: list, act_title: str = ""):
        for node in nodes:
            cur_act = node["title"] if node["node_id"].endswith("::ROOT") else act_title
            if node["node_id"] in target_ids:
                doc_key = node["node_id"].split("::")[0]
                doc_pages = PAGES_STORE.get(doc_key, {})
                s = int(node.get("start_index") or 1)
                e = min(int(node.get("end_index") or s), s + 5)

                # Include next page if a Provison or Explanation crosses a page break
                if re.search(r"(Provided that|Explanation\.|Illustration)", doc_pages.get(str(e), "")[-600:], re.I):
                    if str(e + 1) in doc_pages:
                        e += 1

                text = "\n\n".join(
                    f"[Page {p}]\n{doc_pages[str(p)]}" for p in range(s, e + 1) if str(p) in doc_pages
                )
                hydrated = dict(node)
                hydrated["title"] = f"{cur_act} — {node['title'].strip()}" if cur_act and cur_act != node["title"] else node["title"].strip()
                hydrated["page_index"] = f"{s}-{e}" if s != e else str(s)
                hydrated["text"] = text or "Content not available."
                found.append(hydrated)

            if node.get("nodes"):
                walk(node["nodes"], cur_act)

    walk(tree)
    return found


# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "gemini-3.5-flash-lite") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."

    context_parts = [
        f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
        f"{node.get('text', 'Content not available.')}"
        for node in nodes
    ]
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert legal analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the Act, section title, and page number in parentheses.
    Be concise and precise but thorough in minimum of 3 sentences.

    Question: {query}

    Context:
    {context}

    Answer:"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0),
    )

    return response.text


# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search   → finds relevant node_ids
    Step 2: Node Retrieval    → fetches verbatim statutory pages
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer



In [82]:

# ── Run the full pipeline ────────────────────────────────────────────────────
merged_tree = json.loads((INDEX_DIR / "merged_tree.json").read_text(encoding="utf-8"))

answer = vectorless_rag(
    query="Under BNS 2023 and the constitution of India, what is the punishment for murder, and under what circumstances does the right of private defence of the body extend to causing death?",
    tree=merged_tree
)

🔍 Query: Under BNS 2023 and the constitution of India, what is the punishment for murder, and under what circumstances does the right of private defence of the body extend to causing death?

🧠 Reasoning: 1. Analyze the query: The user is asking about (a) the punishment for murder under the BNS 2023, and (b) the circumstances under which the right of private defence of the body extends to causing death...
🎯 Retrieved node IDs: ['bnsAct2023::0098', 'bnsAct2023::0148', 'bnsAct2023::0149']
📄 Sections found: ['Bharatiya Nyaya Sanhita, 2023 — When right of private defence of body extends to causing death.', 'Bharatiya Nyaya Sanhita, 2023 — CHAPTER VI', 'Bharatiya Nyaya Sanhita, 2023 — Of offences affecting life']

📝 Answer:
Based on the provided context, whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine (Bharatiya Nyaya Sanhita, 2023, Section 103, p. 125). Additionally, when a group of five or more persons acting in concert commits

In [83]:
# ── Run the full pipeline ────────────────────────────────────────────────────
merged_tree = json.loads((INDEX_DIR / "merged_tree.json").read_text(encoding="utf-8"))

answer = vectorless_rag(
    query="I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?",
    tree=merged_tree
)

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

🧠 Reasoning: 1. Analyze the query: The user is asking about a situation where they lost their phone and wallet on a train (traveling across states, Bihar to Delhi) and the local police station is refusing to regis...
🎯 Retrieved node IDs: ['bnssAct2023::0025']
📄 Sections found: ['Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIV']

📝 Answer:
Based on the provided text, because the theft of your phone and wallet occurred while you were performing a journey on a train, the offense may be inquired into or tried by a Court through or into whose local jurisdiction the person or thing passed in the course of that journey (Bharatiya Nagarik Suraksha Sanhita, 2023, Section 203 "Offence committed on journey or voyage," p. 66). Furthermore, regarding offen

In [87]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the fundamental rights in the Constitution of India?",
    "What are the rules for being the judge in the Supreme Court of India?",
    "Summarize the Constitution of India for NRIs.",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, merged_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)
    


Q: What are the fundamental rights in the Constitution of India?
A: Based on the provided context, the Constitution of India guarantees several fundamental rights under Part III (Constitution of India, Part III, pages 37–51):

* **Right to Equality:** Encompasses equality before the law, the prohibition of discrimination on grounds of religion, race, caste, sex, or ...
-------------------------------------------------------

Q: What are the rules for being the judge in the Supreme Court of India?
A: Based on the provided context, there are no specific rules, qualifications, or provisions mentioned regarding the requirements for being a judge in the Supreme Court of India. The text only mentions the Chief Justice of India and the senior-most Judge of the Supreme Court in the context of administe...
-------------------------------------------------------

Q: Summarize the Constitution of India for NRIs.
A: Based on the provided text, the Constitution of India addresses persons of Indian

### Step 7: Expert guided Vectorless RAG Pipeline

---
#### 🎓 Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** - expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain - finance, legal, medical, technical - without any model training.


#### EXPERT RULES

In [88]:
# ── 1. Constitution of India (constitutionOfIndia1950) ───────────────────────
INDIAN_CONSTITUTION_EXPERT_RULES = """
Expert routing rules for the Constitution of India:

- Fundamental Rights, equality, freedom, personal liberty, writs
  → Part III (Articles 12–35)
  → Articles 14–18 (Equality), Articles 19–22 (Freedom, Protection in Conviction, Life & Personal Liberty, Arrest Detention Safeguards), Articles 23–24 (Against Exploitation), Articles 25–28 (Freedom of Religion), Articles 29–30 (Cultural & Educational Rights), Article 32 (Constitutional Remedies).

- Directive Principles, welfare state, social justice, legal aid
  → Part IV (Articles 36–51)
  → Articles 38–39 (Social & Economic Justice), Article 39A (Free Legal Aid), Article 40 (Village Panchayats), Articles 44–47 (Social Welfare), Article 50 (Separation of Judiciary & Executive).

- Fundamental Duties → Part IVA (Article 51A).
- Citizenship → Part II (Articles 5–11).
- Union Executive (President, Vice-President, PM, Council of Ministers, Pardons) → Part V, Chapter I (Articles 52–78; Art. 72 Pardons, Arts. 74–75 Council of Ministers).
- Parliament (Lok Sabha, Rajya Sabha, Money Bills, Budget, Privileges) → Part V, Chapter II (Articles 79–122; Arts. 109–110 Money Bills, Art. 112 Budget).
- Supreme Court (Jurisdiction, SLP, Review, Binding Law, Complete Justice) → Part V, Chapter IV (Articles 124–147; Art. 131 Original, Art. 136 SLP, Art. 141 Precedent, Art. 142 Complete Justice).
- State Executive & Legislature (Governor, CM, State Assembly/Council) → Part VI, Chapters II–III (Articles 153–212).
- High Courts (Writ Jurisdiction & Superintendence) → Part VI, Chapter V (Articles 214–231; Art. 226 Writs, Art. 227 Superintendence).
- Panchayats & Municipalities → Part IX (Articles 243–243O, Eleventh Schedule) & Part IXA (Articles 243P–243ZG, Twelfth Schedule).
- Scheduled & Tribal Areas → Part X (Article 244), Fifth Schedule, Sixth Schedule.
- Centre-State Relations & Legislative Lists → Part XI (Articles 245–263) & Seventh Schedule (Union, State, Concurrent Lists).
- Finance, Taxation, GST Council, Finance Commission, Right to Property → Part XII (Articles 264–300A; Art. 265 Taxation, Art. 279A GST Council, Art. 280 Finance Commission, Art. 300A Property).
- Trade & Commerce → Part XIII (Articles 301–307).
- Public Services & Tribunals → Part XIV (Articles 308–323; Art. 311 Dismissal Safeguards, Art. 315 PSCs) & Part XIVA (Articles 323A–323B).
- Elections & Election Commission → Part XV (Articles 324–329).
- Reservation & National Commissions (SC/ST/BC) → Part XVI (Articles 330–342A).
- Official Languages → Part XVII (Articles 343–351) & Eighth Schedule.
- Emergency Provisions → Part XVIII (Articles 352–360; Art. 352 National, Art. 356 President's Rule, Arts. 358–359 Rights Suspension, Art. 360 Financial).
- Constitutional Amendment → Part XX (Article 368).
- Special State Provisions → Part XXI (Articles 369–392; Arts. 371–371J).
- Anti-Defection → Tenth Schedule (Articles 102(2) and 191(2)).
- Definitions & Interpretation → Part XIX (Articles 366–367) & Part XXII (Articles 393–395).
"""


# ── 2. Bharatiya Nyaya Sanhita, 2023 (bnsAct2023) ────────────────────────────
BNS_EXPERT_RULES = """
Expert routing rules for the Bharatiya Nyaya Sanhita, 2023 (BNS):

- Preliminary, Definitions, Common Intention/Object
  → Chapter I (Sections 1–3)
  → Section 2 (Definitions: electronic record, movable property, public servant, wrongful gain/loss), Section 3(5) (Common intention).

- Punishments, Community Service, Fine, Default Imprisonment
  → Chapter II (Sections 4–13)
  → Section 4 (Death, Life Imprisonment, Rigorous/Simple Imprisonment, Forfeiture, Fine, Community Service), Section 8 (Fine & default).

- General Exceptions, Unsoundness of Mind, Intoxication, Consent, Right of Private Defence
  → Chapter III (Sections 14–44)
  → Sections 14–19 (Mistake of fact, judicial acts, accident, necessity), Sections 20–21 (Child under 7 / 7–12), Section 22 (Unsound mind), Sections 23–24 (Intoxication), Sections 25–30 (Consent), Section 32 (Duress), Section 33 (Trifles),
    Sections 34–44 (Private Defence: s. 37 limits, s. 38 private defence of body extending to death, s. 41 private defence of property extending to death).

- Abetment, Criminal Conspiracy, Attempt
  → Chapter IV (Sections 45–62)
  → Sections 45–60 (Abetment), Section 61 (Criminal conspiracy), Section 62 (Attempt to commit offences).

- Offences Against Woman and Child (Rape, Modesty, Stalking, Dowry Death, Cruelty, Marriage, Child Exploitation)
  → Chapter V (Sections 63–99)
  → Sections 63–72 (Rape, gang rape s. 70, deceitful sexual intercourse s. 69), Sections 74–79 (Outraging modesty, sexual harassment s. 75, voyeurism s. 77, stalking s. 78), Sections 80–87 (Dowry death s. 80, bigamy s. 82, cruelty s. 85), Sections 93–99 (Offences against child).

- Offences Affecting Human Body (Murder, Culpable Homicide, Negligence, Organised Crime, Terrorism, Hurt, Kidnapping)
  → Chapter VI (Sections 100–146)
  → Section 100 (Culpable homicide), Section 101 (Murder & 5 Exceptions), Section 103 (Punishment for murder & mob lynching s. 103(2)), Section 105 (Punishment for culpable homicide), Section 106 (Causing death by negligence & hit-and-run s. 106(2)), Section 108 (Abetment of suicide), Section 109 (Attempt to murder), Section 111 (Organised crime), Section 112 (Petty organised crime), Section 113 (Terrorist act),
    Sections 114–125 (Hurt, grievous hurt, acid attack s. 124), Sections 126–127 (Wrongful restraint & confinement), Sections 128–136 (Criminal force & assault), Sections 137–146 (Kidnapping, abduction, ransom s. 140, human trafficking s. 143).

- Offences Against State, Sovereignty, Armed Forces, Elections, Currency, Public Tranquillity
  → Chapters VII–XI (Sections 147–197)
  → Section 147 (Waging war), Section 152 (Endangering sovereignty, unity & integrity of India), Sections 189–197 (Unlawful assembly s. 189, rioting s. 191, affray s. 194, promoting enmity s. 196).

- Public Servants, Contempt of Lawful Authority, False Evidence & Offences Against Public Justice
  → Chapters XII–XIV (Sections 198–269)
  → Section 221 (Obstructing public servant), Section 223 (Disobedience to public servant order), Sections 227–229 (False evidence/perjury), Section 238 (Causing disappearance of evidence).

- Public Nuisance, Rash Driving, Obscenity, Religion
  → Chapters XV–XVI (Sections 270–302)
  → Section 270 (Public nuisance), Section 281 (Rash driving on public way), Sections 298–302 (Offences relating to religion).

- Offences Against Property (Theft, Snatching, Extortion, Robbery, Dacoity, Criminal Breach of Trust, Cheating, Mischief, Trespass)
  → Chapter XVII (Sections 303–334)
  → Section 303 (Theft), Section 304 (Snatching), Section 308 (Extortion), Section 309 (Robbery), Section 310 (Dacoity), Section 314 (Dishonest misappropriation), Section 316 (Criminal breach of trust), Section 317 (Stolen property), Section 318 (Cheating), Section 319 (Cheating by personation), Section 324 (Mischief), Sections 329–334 (Criminal trespass & house-breaking).

- Forgery, False Documents, Electronic Records, Accounts
  → Chapter XVIII (Sections 335–350)
  → Section 335 (Making false document), Section 336 (Forgery), Section 340 (Using forged document/electronic record).

- Criminal Intimidation, Public Mischief, Defamation, Repeal
  → Chapters XIX–XX (Sections 351–358)
  → Section 351 (Criminal intimidation), Section 353 (Statements conducing to public mischief), Section 356 (Defamation & 10 Exceptions), Section 358 (Repeal of IPC).
"""


# ── 3. Bharatiya Nagarik Suraksha Sanhita, 2023 (bnssAct2023) ────────────────
BNSS_EXPERT_RULES = """
Expert routing rules for the Bharatiya Nagarik Suraksha Sanhita, 2023 (BNSS):

- Preliminary, Definitions (Cognizable/Non-Cognizable, Bailable/Non-Bailable, Electronic Communication) & Court Powers
  → Chapters I–III (Sections 1–29)
  → Section 2 (Definitions), Sections 21–23 (Trial jurisdiction & sentencing powers of Courts/Magistrates).

- Arrest of Persons, Notice of Appearance, Rights of Arrested Person, Handcuffing, 24-Hour Rule
  → Chapter V (Sections 35–62)
  → Section 35 (Arrest without warrant; s. 35(3) Notice of appearance; s. 35(7) prior permission for infirm/60+ in <3 yr offences), Section 36 (Arrest procedure), Section 38 (Right to meet advocate during interrogation), Section 43 (Arrest how made, handcuffing s. 43(3), night arrest of women s. 43(5)), Section 47 (Grounds of arrest & right to bail), Section 48 (Inform nominated person), Section 57 (24-hour detention limit).

- Summons, Electronic Summons, Warrants, Proclamation & Attachment of Property
  → Chapters VI–VII (Sections 63–110)
  → Sections 63–71 (Summons & electronic service), Sections 72–83 (Warrants), Sections 84–89 (Proclamation & attachment of absconder), Section 94 (Summons to produce documents/electronic records), Section 105 (Mandatory audio-video recording of search & seizure), Section 107 (Attachment/forfeiture of proceeds of crime).

- Maintenance of Wives, Children and Parents
  → Chapter X (Sections 144–147) → Section 144 (Maintenance order).

- Public Order, Urgent Nuisance/Danger Orders, Preventive Detention
  → Chapters XI–XII (Sections 148–172)
  → Section 152 (Conditional order for removal of nuisance), Section 163 (Urgent orders in apprehended danger), Sections 170–172 (Preventive arrest & 24-hour detention for lawful police directions).

- FIR, Zero FIR, e-FIR, Police Investigation, Forensics, Remand, Default Bail, Chargesheet
  → Chapter XIII (Sections 173–196)
  → Section 173 (Cognizable FIR, Zero FIR, e-FIR, 14-day preliminary inquiry for 3–7 yr offences), Section 174 (Non-cognizable info), Section 175 (Power to investigate; s. 175(3) Magistrate direction), Section 176 (Investigation procedure; s. 176(3) mandatory forensic visit for >=7 yr offences), Sections 179–180 (Witness attendance & examination), Section 183 (Magistrate recording of confessions/statements), Section 184 (Medical exam of rape victim), Section 185 (Police search with audio-video recording), Section 187 (Police/Judicial custody remand & 60/90-day default bail), Section 193 (Chargesheet within 90 days & victim update).

- Cognizance, Private Complaints, Supply of Documents, Framing of Charge, Trial & Plea Bargaining
  → Chapters XV–XXIII (Sections 210–300)
  → Section 210 (Cognizance), Section 218 (Sanction to prosecute public servants), Section 223 (Complaint examination & accused hearing proviso), Section 230 (Supply of documents within 14 days), Sections 234–247 (Charge), Sections 248–288 (Sessions, Warrant, Summons & Summary trials), Sections 289–300 (Plea bargaining).

- General Trial Provisions, Legal Aid, Trial in Absentia, Victim Compensation, Compounding, Judgment
  → Chapters XXVI–XXIX (Sections 337–406)
  → Section 337 (Double jeopardy bar), Section 341 (Free legal aid), Section 356 (Trial in absentia of proclaimed offender), Section 358 (Victim compensation), Section 359 (Compounding of offences), Section 392 (Judgment within 45 days), Section 398 (Witness protection).

- Appeals, Revision, Transfer, Sentence Suspension/Mercy Petitions
  → Chapters XXX–XXXIV (Sections 407–477)
  → Section 413 (Appeals & victim appeal proviso), Sections 438 & 442 (Revision), Section 472 (Mercy petition), Sections 473–475 (Suspension/remission/commutation).

- Bail, Undertrial Detention Limit, Anticipatory Bail
  → Chapter XXXV (Sections 478–496)
  → Section 478 (Bail in bailable offences), Section 479 (Maximum undertrial detention: 1/2 max term; 1/3 for first-time offender), Section 480 (Bail in non-bailable offences), Section 482 (Anticipatory bail), Section 483 (High Court / Sessions Court special bail powers).

- Inherent Powers of High Court, Electronic Proceedings, Repeal of CrPC
  → Chapters XXXVIII–XXXIX (Sections 513–531)
  → Sections 513–519 (Limitation period), Section 528 (Inherent powers of High Court), Section 530 (Electronic trials/proceedings), Section 531 (Repeal of CrPC, 1973).
"""


# ── 4. Digital Personal Data Protection Act, 2023 (dpdpAct2023) ──────────────
DPDP_EXPERT_RULES = """
Expert routing rules for the Digital Personal Data Protection Act, 2023 (DPDP Act):

- Applicability, Extraterritorial Scope, Exemptions (Domestic/Publicly Available Data) & Definitions
  → Chapter I (Sections 1–3)
  → Section 2 (Definitions: Data Fiduciary, Data Principal, Data Processor, Significant Data Fiduciary, Consent Manager, Personal Data Breach), Section 3 (Digital/digitized personal data in India, foreign processing for Indian goods/services, domestic & public data exemptions).

- Lawful Grounds, Notice, Consent, Withdrawal, Legitimate Uses, Fiduciary Obligations, Children's Data, Significant Data Fiduciaries
  → Chapter II (Sections 4–10)
  → Section 4 (Lawful purpose: consent or legitimate uses), Section 5 (Itemised notice), Section 6 (Free, specific, informed, unambiguous consent, withdrawal & Consent Manager), Section 7 (Legitimate uses: voluntary data, State subsidies/services, sovereignty/security, legal compliance, medical emergency, disaster, employment),
    Section 8 (Fiduciary duties: accuracy, reasonable security safeguards, breach notification to Board & Data Principal, data erasure, DPO/contact, grievance redressal), Section 9 (Children's data: verifiable parental consent, ban on tracking/behavioural monitoring/targeted ads), Section 10 (Significant Data Fiduciary: India-based DPO, Independent Data Auditor, DPIA).

- Rights and Duties of Data Principal
  → Chapter III (Sections 11–15)
  → Section 11 (Right to access info), Section 12 (Right to correction & erasure), Section 13 (Right of grievance redressal), Section 14 (Right to nominate), Section 15 (Duties of Data Principal).

- Cross-Border Data Transfers & Statutory Exemptions (Law Enforcement, Courts, Research, Startups)
  → Chapter IV (Sections 16–17)
  → Section 16 (Cross-border transfer allowed unless restricted by Central Govt notification), Section 17 (Exemptions: legal claims/courts, offence investigation/prosecution, foreign outsourcing contracts, mergers, State security/public order, research/statistics, startups).

- Data Protection Board of India, Powers, Inquiry Procedure, Appeals (TDSAT), Mediation & Penalties
  → Chapters V–IX & The Schedule (Sections 18–44)
  → Sections 18–26 (Establishment & composition of Board), Sections 27–28 (Board powers, breach directions & digital procedure), Sections 29–32 (Appeal to TDSAT within 60 days, ADR/mediation, voluntary undertaking), Sections 33–34 & The Schedule (Financial penalties up to Rs. 250 Cr for security safeguard failure, Rs. 200 Cr for breach non-notification or children's data violation, Rs. 150 Cr for SDF default), Section 37 (Blocking directions), Section 38 (Overriding effect), Section 43 (Amendments to IT Act & RTI Act s. 8(1)(j)).
"""


# ── 5. Information Technology Act, 2000 (itAct2020) ──────────────────────────
IT_ACT_EXPERT_RULES = """
Expert routing rules for the Information Technology Act, 2000 (IT Act):

- Applicability, Extraterritorial Cyber Jurisdiction & Definitions
  → Chapter I (Sections 1–2)
  → Section 1(2) & Section 75 (Offences committed outside India involving computer/network in India), Section 2 (Definitions: computer resource, cyber security, digital/electronic signature, electronic record, intermediary, secure system).

- Digital/Electronic Signatures, E-Governance, Electronic Contracts & Attribution
  → Chapters II–V (Sections 3–16)
  → Sections 3–3A (Digital & electronic signatures), Sections 4–5 (Legal recognition of electronic records & signatures), Sections 6–7 (E-governance & record retention), Section 10A (Validity of e-contracts), Sections 11–13 (Attribution, acknowledgment, despatch/receipt).

- Certifying Authorities, Electronic Signature Certificates & Subscriber Duties
  → Chapters VI–VIII (Sections 17–42)
  → Section 17 (Controller of Certifying Authorities), Sections 21–30 (CA licensing & procedures), Sections 35–39 (Certificates), Sections 40–42 (Private key control & subscriber duties).

- Civil Cyber Contraventions, Compensation, Adjudication & Appellate Tribunal (TDSAT)
  → Chapters IX–X (Sections 43–62)
  → Section 43 (Compensation/penalty for unauthorized access, data theft, virus/contaminant, disruption, denial of access, source code theft), Section 43A (Compensation for failure to protect sensitive personal data), Section 46 (Adjudicating Officer), Sections 48–62 (Appellate Tribunal / TDSAT & High Court appeals).

- Cyber Offences, Privacy Violation, Cyber Terrorism, Obscenity, Interception, Website Blocking, CERT-In & Investigation
  → Chapter XI (Sections 65–78)
  → Section 65 (Tampering with computer source documents), Section 66 (Computer-related offences), Section 66B (Stolen computer/device), Section 66C (Identity theft), Section 66D (Cheating by personation via computer), Section 66E (Violation of bodily privacy), Section 66F (Cyber terrorism),
    Sections 67, 67A, 67B (Obscene, sexually explicit & child sexually explicit electronic material), Section 69 (Interception/monitoring/decryption), Section 69A (Blocking public access to online information), Section 69B (Traffic data monitoring), Sections 70–70B (Protected systems, NCIIPC, CERT-In), Sections 72 & 72A (Breach of confidentiality/privacy & lawful contract), Section 78 (Investigation by Inspector rank & above).

- Intermediary Safe Harbour, Examiner of Electronic Evidence, Search & Overriding Effect
  → Chapters XII–XIII (Sections 79–90)
  → Section 79 (Intermediary exemption from liability / safe harbour & due diligence), Section 79A (Examiner of Electronic Evidence), Section 80 (Power of police/officers to enter & search), Section 81 (Overriding effect subject to Copyright/Patents Acts), Section 85 (Offences by companies).
"""


# ── 6. Cross-Statute Legal Interpretation Rules (Appended to Active Rules) ───
SHARED_LEGAL_INTERPRETATION_RULES = """
IMPORTANT LEGAL INTERPRETATION RULES:
1. Route questions to the exact Act, Part, Chapter, Section, Article, or Schedule that legally governs the issue.
2. When a query spans substantive crime + criminal procedure (e.g., murder/theft + arrest/bail/FIR), retrieve from BOTH BNS 2023 and BNSS 2023.
3. When a query involves cybercrime, online platforms, or personal data breaches, check intersections across IT Act 2000, DPDP Act 2023, BNS 2023, and Article 21 of the Constitution.
4. Prefer specific statutory provisions over general provisions when both address the same issue.
5. Always read provisions together with their Provisos, Explanations, Illustrations, General Exceptions (BNS Ch. III), and Statutory Exemptions (DPDP s. 17).
6. Distinguish Fundamental Rights (Part III) from Directive Principles (Part IV, Art. 37) and Fundamental Duties (Part IVA).
7. Use current amended provisions (BNS/BNSS rather than repealed IPC/CrPC references in comparative tables) and never attribute rules from one statute to another.
8. Do not infer a legal power, right, restriction or obligation unless supported by an applicable constitutional provision or binding judicial interpretation.
9. Do not assume that every constitutional right is absolute. Examine applicable restrictions and exceptions.
10. If the Constitution does not directly address a question,identify the relevant ordinary legislation, rules or judicial precedent separately. Do not attribute their provisions to the Constitution itself.

"""

print("✅ Expert rules defined for guided Retrieval!")

✅ Expert rules defined for guided Retrieval!


In [ ]:
STATUTE_RULE_REGISTRY = {
    "constitution": (
        {"constitution", "article", "fundamental right", "writ", "parliament", "president",
         "governor", "supreme court", "high court", "directive principle", "panchayat",
         "amendment", "emergency", "citizenship", "schedule", "election commission"},
        INDIAN_CONSTITUTION_EXPERT_RULES,
    ),
    "bns": (
        {"bns", "bharatiya nyaya sanhita", "penal", "murder", "culpable homicide", "theft",
         "snatching", "robbery", "dacoity", "extortion", "cheating", "forgery", "defamation",
         "private defence", "abetment", "conspiracy", "rape", "dowry", "hurt", "kidnapping",
         "organised crime", "terrorist", "sedition", "sovereignty"},
        BNS_EXPERT_RULES,
    ),
    "bnss": (
        {"bnss", "bharatiya nagarik suraksha", "fir", "zero fir", "refusal to fir", "arrest", "bail",
         "anticipatory bail", "remand", "custody", "chargesheet", "investigation", "magistrate",
         "summons", "warrant", "proclamation", "search", "seizure", "plea bargaining",
         "maintenance", "cognizance", "trial", "appeal", "revision", "mercy petition"},
        BNSS_EXPERT_RULES,
    ),
    "dpdp": (
        {"dpdp", "digital personal data", "data fiduciary", "data principal", "data processor",
         "significant data fiduciary", "consent manager", "personal data", "data breach",
         "data protection board", "parental consent", "erasure", "nomination"},
        DPDP_EXPERT_RULES,
    ),
    "it_act": (
        {"it act", "information technology", "cyber", "intermediary", "safe harbour",
         "digital signature", "electronic signature", "electronic record", "certifying authority",
         "identity theft", "hacking", "computer resource", "blocking", "interception",
         "cert-in", "tampering", "66a", "66c", "66d", "66e", "67", "69a", "79"},
        IT_ACT_EXPERT_RULES,
    ),
}


def select_expert_rules(query: str) -> str:
    """Inject only the relevant statute routing rules for the query (keeps prompt fast & focused)."""
    q_low = query.lower()
    matched_blocks = [
        rules_text
        for _, (keywords, rules_text) in STATUTE_RULE_REGISTRY.items()
        if any(kw in q_low for kw in keywords)
    ]

    # If query uses broad everyday phrasing (e.g., "police detained me without warrant"),
    # include Constitution + BNS + BNSS rules so cross-statute rights & procedures are covered.
    if not matched_blocks:
        matched_blocks = [
            INDIAN_CONSTITUTION_EXPERT_RULES,
            BNS_EXPERT_RULES,
            BNSS_EXPERT_RULES,
        ]

    return "\n".join(matched_blocks) + "\n" + SHARED_LEGAL_INTERPRETATION_RULES

print("👩🏻‍🏫 Expert rules get selected based on the retrieval prompt at query time to stay within token limit!")

👩🏻‍🏫 Expert rules get selected based on the retrieval prompt at query time to stay within token limit!


#### MASTER RULES (combining all rules for combined better retrieval across all legal documents)

Nodes retrieval using Expert Rules

In [111]:
MASTER_EXPERT_RULES = "\n\n".join([
    INDIAN_CONSTITUTION_EXPERT_RULES.strip(),
    BNS_EXPERT_RULES.strip(),
    BNSS_EXPERT_RULES.strip(),
    DPDP_EXPERT_RULES.strip(),
    IT_ACT_EXPERT_RULES.strip(),
    SHARED_LEGAL_INTERPRETATION_RULES.strip(),
])

In [116]:
import json
import re
from pathlib import Path
from google import genai
from google.genai import types

INDEX_DIR = Path("./legal_index") if Path("./legal_index").exists() else Path("../legal_index")
gemini_client = globals().get("gemini_client") or genai.Client()

_manifest = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
PAGES_STORE = {
    k: json.loads((INDEX_DIR / Path(v["pages_file"]).name).read_text(encoding="utf-8"))
    for k, v in _manifest.items()
}

EXCLUDED_CONTAINERS = {"homepage", "index", "compare", "arrangement of sections"}
MIN_BODY_PAGE = {"constitutionOfIndia1950": 32,
                #   "bnsAct2023": 21, "bnssAct2023": 21
                  }

In [117]:
# ── Step 1: Rule-First Expert Tree Search (for merged_tree list) ─────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str = MASTER_EXPERT_RULES,
    model: str = "gemini-3.5-flash-lite",
) -> dict:
    """
    Two-stage expert-guided tree search over merged_tree (list of Act dicts):
    1. Decomposes the query into all legal issues using Expert Routing Rules,
       mapping Roman-numeral Chapters/Parts (e.g., 'CHAPTER XIII') to exact node_ids.
    2. Drills into child sections only if the selected chapters have child nodes.
    """
    active_rules = expert_rules or (
        select_expert_rules(query) if "select_expert_rules" in globals() else ""
    )

    def compress(nodes: list, depth: int = 0, max_depth: int = 1, valid_ids: set[str] | None = None) -> list:
        out = []
        for n in nodes:
            title = n.get("title", "").strip()
            title_low = title.lower()
            doc_key = n["node_id"].split("::")[0]
            start_p = int(n.get("start_index") or 1)

            if title_low in EXCLUDED_CONTAINERS or re.match(r"^form\s+no\.?", title_low):
                continue
            if not n["node_id"].endswith("::ROOT") and start_p < MIN_BODY_PAGE.get(doc_key, 1):
                continue
            if title_low == "chapters" and n.get("nodes"):
                out.extend(compress(n["nodes"], depth, max_depth, valid_ids))
                continue

            nid = n["node_id"]
            if valid_ids is not None:
                valid_ids.add(nid)

            entry = {
                "node_id": nid,
                "title": title,
                "pages": f"{start_p}-{n.get('end_index', '?')}",
            }
            if n.get("summary"):
                entry["summary"] = n["summary"]
            elif n.get("key_items"):
                items = [k for k in n["key_items"] if not k.lower().startswith("p.")]
                if items:
                    entry["key_items"] = items[:12]

            if depth < max_depth and n.get("nodes"):
                entry["children"] = compress(n["nodes"], depth + 1, max_depth, valid_ids)

            out.append(entry)
        return out

    def query_gemini(tree_slice: list, stage_instruction: str, valid_ids: set[str]) -> dict:
        prompt = f"""You are an Indian legal routing specialist.
{stage_instruction}

CRITICAL INSTRUCTIONS:
1. Break the user's query into ALL distinct legal issues (for example: police duty/procedure vs. court jurisdiction, or substantive offence vs. bail/arrest).
2. Match EVERY issue against the Expert Routing Rules below to identify the exact Act, Chapter/Part (including Roman numerals like CHAPTER XIII), and Section/Article numbers.
3. Map every matched Chapter/Part to its exact node_id in the Document Tree—even if the tree node title only shows a Roman numeral (e.g., "CHAPTER XIII") without a subtitle.
4. Select ALL relevant node_ids (up to 4), not just one.

Query: {query}

Expert Routing Rules:
{active_rules}

Document Tree:
{json.dumps(tree_slice, separators=(",", ":"), ensure_ascii=False)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<list each legal issue -> matched Expert Rule (Chapter + Section numbers) -> exact node_id>",
  "node_list": ["docKey::node_id1", "docKey::node_id2"]
}}"""

        response = gemini_client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type="application/json",
            ),
        )
        raw = (response.text or "{}").strip()
        match = re.search(r"[\{\[].*[\}\]]", raw, flags=re.S)
        parsed = json.loads(match.group(0) if match else raw)
        if isinstance(parsed, str):
            parsed = json.loads(parsed)
        if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
            parsed = parsed[0]
        if not isinstance(parsed, dict):
            parsed = {"thinking": "", "node_list": []}

        parsed["node_list"] = [nid for nid in parsed.get("node_list", []) if nid in valid_ids]
        return parsed

    # Stage 1: Select all relevant Chapters/Parts across merged_tree
    stage1_ids: set[str] = set()
    coarse_tree = compress(tree, depth=0, max_depth=1, valid_ids=stage1_ids)
    stage1 = query_gemini(
        coarse_tree,
        "Select up to 4 Chapter or Part node_ids covering every legal issue in the query.",
        stage1_ids,
    )
    chapter_ids = stage1["node_list"][:4]
    if not chapter_ids:
        return stage1

    # Stage 2: Drill down ONLY if the chosen chapters actually have child nodes
    def extract_subtrees(nodes: list, targets: set[str]) -> list:
        found = []
        for n in nodes:
            if n["node_id"] in targets:
                found.append(n)
            elif n.get("nodes"):
                found.extend(extract_subtrees(n["nodes"], targets))
        return found

    subtrees = extract_subtrees(tree, set(chapter_ids))
    has_children = any(
        c.get("title", "").strip().lower() not in EXCLUDED_CONTAINERS
        for st in subtrees
        for c in st.get("nodes", [])
    )
    if not has_children:
        return {
            "thinking": stage1.get("thinking", ""),
            "node_list": chapter_ids,
        }

    stage2_ids: set[str] = set()
    fine_tree = compress(subtrees, depth=0, max_depth=3, valid_ids=stage2_ids)
    stage2 = query_gemini(
        fine_tree,
        "Select up to 4 most specific leaf/section node_ids (or keep parent chapter IDs for flat chapters).",
        stage2_ids,
    )
    final_nodes = stage2["node_list"][:4] or chapter_ids
    return {
        "thinking": f"[Stage 1] {stage1.get('thinking', '')}\n[Stage 2] {stage2.get('thinking', '')}",
        "node_list": final_nodes,
    }


In [118]:
# ── Step 2: Section-Aware Node & Page Retrieval ──────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list, hint_text: str = "") -> list:
    """
    Walk merged_tree (list), collect target_ids, and hydrate verbatim pages from *_pages.json.
    Prioritizes pages matching Section/Article numbers cited in hint_text.
    """
    target_set = set(target_ids)
    target_nums = set(re.findall(r"\b(?:section|sec\.?|article|art\.?|s\.)\s*(\d+[A-Z]?)\b", hint_text, flags=re.I))
    matched: dict[str, dict] = {}

    def walk(nodes: list, act_title: str = ""):
        for node in nodes:
            cur_act = node["title"] if node["node_id"].endswith("::ROOT") else act_title
            nid = node["node_id"]

            if nid in target_set and nid not in matched:
                doc_key = nid.split("::")[0]
                doc_pages = PAGES_STORE.get(doc_key, {})
                s = int(node.get("start_index") or 1)
                raw_e = int(node.get("end_index") or s)

                hit_pages = []
                if target_nums and (raw_e - s) > 5:
                    for p in range(s, raw_e + 1):
                        p_txt = doc_pages.get(str(p), "")
                        if any(re.search(rf"(?:^|\n)\s*{num}\.\s", p_txt) for num in target_nums):
                            hit_pages.extend([p, p + 1])

                if hit_pages:
                    selected_pages = sorted({p for p in hit_pages if s <= p <= raw_e})[:6]
                else:
                    e = min(raw_e, s + 5)
                    if re.search(r"(Provided that|Explanation\.|Illustration)", doc_pages.get(str(e), "")[-600:], re.I):
                        if str(e + 1) in doc_pages:
                            e += 1
                    selected_pages = [p for p in range(s, e + 1) if str(p) in doc_pages]

                text = "\n\n".join(f"[Page {p}]\n{doc_pages[str(p)]}" for p in selected_pages if str(p) in doc_pages)
                hydrated = dict(node)
                hydrated["title"] = f"{cur_act} — {node['title'].strip()}" if cur_act and cur_act != node["title"] else node["title"].strip()
                hydrated["page_index"] = f"{selected_pages[0]}-{selected_pages[-1]}" if len(selected_pages) > 1 else (str(selected_pages[0]) if selected_pages else "?")
                hydrated["text"] = text or "Content not available."
                matched[nid] = hydrated

            if node.get("nodes"):
                walk(node["nodes"], cur_act)

    walk(tree)
    return [matched[nid] for nid in target_ids if nid in matched]

### Step 9 - Comparing Node Selection in both with and without Expert Guidance

Check the Node selection using Vectorless pipeline (without expert rules) and Vectorless pipeline (with expert rules of `INDIAN_CONSTITUTION_EXPERT_RULES`)

#### **Some important nodes were missing because only 1 expert rule set was passed.**

In [119]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "What is the punishment for Human Trafficking under the Constitution of India?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, merged_tree)
print("Nodes:", basic.get("node_list"))
print("Reasoning:", basic.get("thinking", "")[:300])

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, merged_tree, INDIAN_CONSTITUTION_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: What is the punishment for Human Trafficking under the Constitution of India?

── Without Expert Rules ──
Nodes: ['constitutionOfIndia1950::ROOT', 'constitutionOfIndia1950::0039', 'constitutionOfIndia1950::0043']
Reasoning: The query asks about the punishment for Human Trafficking under the Constitution of India. However, the Constitution of India contains fundamental rights like the Right against Exploitation (Article 23 prohibits human trafficking), but penal punishments (criminal offences) are typically defined in c

── With Expert Rules ──
Nodes: ['constitutionOfIndia1950::0043']
Reasoning: [Stage 1] Issue 1: Human Trafficking (Prohibition against traffic in human beings and forced labor) -> Fundamental Rights, Against Exploitation -> Constitution of India Part III (Articles 23-24) -> constitutionOfIndia1950::0039.
[Stage 2] Human Trafficking -> Constitution of India, Part III, Article


Check the Node selection using Vectorless pipeline (without expert rules) and Vectorless pipeline (with expert rules of `MASTER_EXPERT_RULES` combined all)

#### **Some important nodes were added because all the expert rule set were passed.**

In [124]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, merged_tree)
print("Nodes:", basic.get("node_list"))
print("Reasoning:", basic.get("thinking", "")[:])

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, merged_tree, MASTER_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:])

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

── Without Expert Rules ──
Nodes: ['bnssAct2023::0025']
Reasoning: 1. Analyze the user's query: The user lost their phone and wallet on a train while traveling from Bihar to Delhi, and the local police station refused to register an FIR claiming lack of territorial jurisdiction. The user wants to know what the law requires the police to do in such cases.
2. Identify the legal domain: This relates to criminal procedure in India, specifically the registration of First Information Reports (FIRs) and police jurisdiction, which falls under the Bharatiya Nagarik Suraksha Sanhita (BNSS), 2023 (the replacement for the Code of Criminal Procedure).
3. Scan the Document Tree for relevant chapters/sections:
   - `bnssAct2023::ROOT`: Bharatiya Nagarik Suraksha San

In [125]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "I lost my phone and wallet on a train while traveling from Delhi to Kerala, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, merged_tree)
print("Nodes:", basic.get("node_list"))
print("Reasoning:", basic.get("thinking", "")[:])

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, merged_tree, MASTER_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:])

🔍 Query: I lost my phone and wallet on a train while traveling from Delhi to Kerala, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

── Without Expert Rules ──
Nodes: ['bnssAct2023::0025']
Reasoning: The query asks about the legal obligation of a local police station when someone attempts to file an FIR for an incident that occurred outside their jurisdiction (Zero FIR concept under criminal procedure law in India). Looking at the document tree, the relevant legislation governing criminal procedure, investigations, and police duties is the Bharatiya Nagarik Suraksha Sanhita, 2023 (BNSS). Specifically, Chapter XIV deals with the jurisdiction of criminal courts and investigations, and Chapter V/XIII/XIV cover police powers and procedures. Let's look for nodes under BNSS Act 2023 that cover jurisdiction, police reports, or general provisions regarding inquiries and trials, such as CH

In [ ]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Under BNS 2023 and the constitution of India, what is the punishment for murder, and under what circumstances does the right of private defence of the body extend to causing death?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, merged_tree)
print("Nodes:", basic.get("node_list"))
print("Reasoning:", basic.get("thinking", "")[:300])

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, merged_tree, MASTER_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [126]:
# ── Step 3: Full Expert-Guided RAG Orchestrator ──────────────────────────────

def expert_rag(
    query: str,
    tree: list,
    rules: str = MASTER_EXPERT_RULES,
    verbose: bool = True,
) -> str:
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Expert-guided Tree Search (auto-selects cross-Act rules if rules=None)
    result = llm_tree_search_with_expert(query, tree, expert_rules=rules)
    node_ids = result.get("node_list", [])
    thinking = result.get("thinking", "")

    if verbose:
        print(f"\n🧠 Reasoning: {thinking[:250]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve and hydrate nodes from *_pages.json
    nodes = find_nodes_by_ids(tree, node_ids, hint_text=thinking)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate grounded answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

### Step 10 : Testing the final answers of both RAG pipelines

In [ ]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def with_expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    print("Expert RAG says: ")
    return expert_rag(query, tree, rules)

# Run with explicit Constitution rules
answer = with_expert_rag(
    query="I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?",
    tree=merged_tree,
    rules=INDIAN_CONSTITUTION_EXPERT_RULES,
)

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

🧠 Reasoning: [Stage 1 — Chapters] The query deals with a situation where a person lost property (phone and wallet) while traveling on a train across states (Bihar to Delhi) and the local police station is refusing...
🎯 Retrieved node IDs: ['bnssAct2023::0025']
📄 Sections found: ['Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIV']

📝 Answer:
Based on the provided context, because the theft of your phone and wallet occurred while you were performing a journey on a train, the offense may be inquired into or tried by a Court through or into whose local jurisdiction the person or thing passed in the course of that journey (Bharatiya Nagarik Suraksha Sanhita, 2023, Section 203 "Offence committed on journey or voyage", p. 66). Furthermore, any offence 

In [ ]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def with_expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    print("Expert RAG says: ")
    return expert_rag(query, tree, rules)

# Run with explicit Constitution rules
answer = with_expert_rag(
    query="What is the punishment for Human Trafficking under the Constitution of India?",
    tree=merged_tree,
    rules=INDIAN_CONSTITUTION_EXPERT_RULES,
)

In [108]:
answer = expert_rag(
    query="I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?",
    tree=merged_tree,
    rules=BNSS_EXPERT_RULES, #INDIAN_CONSTITUTION_EXPERT_RULES
)

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

🧠 Reasoning: 1. Issue: Refusal of local police station to register an FIR based on jurisdiction (Zero FIR / cognizable offence reporting) -> Bharatiya Nagarik Suraksha Sanhita, 2023 (BNSS) Chapter XIII, Section 173 -> bnssAct2023::0024. 2. Issue: Territorial juri...
🎯 Retrieved node IDs: ['bnssAct2023::0024', 'bnssAct2023::0025']
📄 Sections found: ['Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIII', 'Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIV']

📝 Answer:
Under the Bharatiya Nagarik Suraksha Sanhita, 2023, information relating to the commission of any cognizable offence must be given to an officer in charge of a police station "irrespective of the area where the offence is committed" (Section 173(1), Chapter XIII, p. 52). Therefore,

In [113]:
answer = expert_rag(
    query="I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?",
    tree=merged_tree,
    rules=MASTER_EXPERT_RULES,
)

🔍 Query: I lost my phone and wallet on a train while traveling from Bihar to Delhi, and my local police station is refusing to register an FIR saying the incident happened outside their jurisdiction. What does the law require them to do?

🧠 Reasoning: [Stage 1] 1. Issue: Refusal of local police station to register an FIR based on territorial jurisdiction -> Matched Expert Rule: BNSS Chapter XIII (Sections 173-196, specifically Section 173 regarding Cognizable FIR and Zero FIR) -> node_id: bnssAct2...
🎯 Retrieved node IDs: ['bnsAct2023::0266', 'bnssAct2023::0024', 'bnssAct2023::0025']
📄 Sections found: ['Bharatiya Nyaya Sanhita, 2023 — CHAPTER XVII', 'Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIII', 'Bharatiya Nagarik Suraksha Sanhita, 2023 — CHAPTER XIV']

📝 Answer:
Under the law, every information relating to the commission of a cognizable offence must be given to an officer in charge of a police station "irrespective of the area where the offence is committed" (Bharatiya Nag

In [127]:
answer = expert_rag(
    query="Two men on a motorcycle snatched my mobile phone from my hand while I was walking on the street and sped away without hurting me. Is this ordinary theft or a separate offence under BNS 2023, and what is the punishment?",
    tree=merged_tree,
    rules=MASTER_EXPERT_RULES,
)

🔍 Query: Two men on a motorcycle snatched my mobile phone from my hand while I was walking on the street and sped away without hurting me. Is this ordinary theft or a separate offence under BNS 2023, and what is the punishment?

🧠 Reasoning: [Stage 1] 1. Issue 1: Substantive offence of mobile snatching (theft vs. snatching under BNS 2023) -> Bharatiya Nyaya Sanhita, 2023, Chapter XVII (Sections 303 & 304 - Theft and Snatching) -> bnsAct2023::0266
2. Issue 2: General exceptions or definit...
🎯 Retrieved node IDs: ['bnsAct2023::0266', 'bnsAct2023::0268', 'bnsAct2023::0269']
📄 Sections found: ['Bharatiya Nyaya Sanhita, 2023 — CHAPTER XVII', 'Bharatiya Nyaya Sanhita, 2023 — Theft.', 'Bharatiya Nyaya Sanhita, 2023 — Snatching.; Theft in a dwelling house, or means of transportation or place of worship, etc.']

📝 Answer:
Based on the provided context, the act of two men suddenly or quickly taking your mobile phone from your hand constitutes a separate offence known as snatching rather than or